# 📙 Notebook 3 — Model Architectures
## Baseline Models + Proposed BERT-CNN Hybrid

This notebook defines all model architectures used in the research:

| # | Model | Type | Parameters |
|---|-------|------|------------|
| 1 | BERT | Baseline | 110M |
| 2 | RoBERTa | Baseline | 125M |
| 3 | DistilBERT | Baseline | 66M |
| 4 | **BERT-CNN** | **Proposed** | **~113M** |

> **Novel contribution:** The BERT-CNN hybrid captures both *global* document semantics (via BERT's [CLS] token) and *local* phrase-level signals (via multi-scale 1D CNN with attention gates), fused via a residual connection and trained with Focal Loss.

In [ ]:
import os, torch, warnings
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel
warnings.filterwarnings('ignore')

# Config
DEVICE           = "cuda" if torch.cuda.is_available() else "cpu"
CNN_FILTERS      = 256
CNN_KERNEL_SIZES = [2, 3, 4]
DROPOUT_RATE     = 0.3
MODELS = {
    "BERT":       "bert-base-uncased",
    "RoBERTa":    "roberta-base",
    "DistilBERT": "distilbert-base-uncased",
    "BERT-CNN":   "bert-base-uncased",
}
print("Device:", DEVICE)

## 3.1 Baseline Classifier
### Architecture
```
Input Text → [Tokenizer] → BERT/RoBERTa/DistilBERT Encoder → [CLS] token (768-dim)
           → Dropout(0.3) → Linear(768 → 2) → Logits
```
**Loss:** Weighted Cross-Entropy (handles class imbalance)

This single class works for all three baseline models. The forward pass auto-detects whether the model exposes a `pooler_output` (BERT, RoBERTa) or falls back to the raw [CLS] hidden state (DistilBERT).

In [ ]:
class BaselineClassifier(nn.Module):
    """
    Standard transformer fine-tuning classifier.
    Compatible with BERT, RoBERTa, and DistilBERT via AutoModel.
    Architecture:
        Encoder → [CLS] pooled output → Dropout → Linear(hidden → 2)
    """
    def __init__(self, model_name: str, num_classes: int = 2,
                 dropout: float = DROPOUT_RATE):
        super().__init__()
        self.encoder    = AutoModel.from_pretrained(model_name)
        hidden_size     = self.encoder.config.hidden_size
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, num_classes)

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        out = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
        )
        # BERT/RoBERTa → pooler_output  |  DistilBERT → last_hidden_state[:,0]
        if hasattr(out, "pooler_output") and out.pooler_output is not None:
            pooled = out.pooler_output
        else:
            pooled = out.last_hidden_state[:, 0, :]

        return self.classifier(self.dropout(pooled))


# Quick parameter count
def count_params(model):
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

print("BaselineClassifier defined.")
print("\nParameter counts:")
for name, ckpt in [("BERT","bert-base-uncased"),("RoBERTa","roberta-base"),("DistilBERT","distilbert-base-uncased")]:
    m = BaselineClassifier(ckpt)
    t, tr = count_params(m)
    print(f"  {name:<12}: {t/1e6:.1f}M total | {tr/1e6:.1f}M trainable")
    del m

## 3.2 Proposed Model: BERT-CNN Hybrid
### Architecture Overview
```
Input Text
   │
   ▼
BERT Encoder (bert-base-uncased, 12 layers)
   │
   ├── [CLS] token ──────────────────────────► (B, 768)   ← Global context
   │
   └── All token embeddings (B, 512, 768)
              │
              ▼
       Multi-Scale CNN  [k=2, k=3, k=4 in parallel]
              │
       Attention Gate per kernel   ← learns which n-grams matter
              │
       Concat → (B, 768)           ← Local phrase-level features
              │
              ▼
   Residual Fusion: [CLS ⊕ CNN] → LayerNorm → (B, 1536)
              │
              ▼
   MLP Head:  1536 → 512 → 128 → 2
              │
              ▼
   Logits (B, 2)   ← Trained with Focal Loss
```

### Why this works for mental health text
- Reddit posts about suicidal ideation often contain **short, emotionally charged phrases** like *"want to die"*, *"end it all"*, *"can't take it anymore"*
- Standard [CLS]-only BERT may dilute these signals in very long posts (512 tokens)
- The **multi-scale CNN** specifically targets 2–4 word windows, capturing these exact patterns
- The **attention gate** learns to up-weight the most indicative n-gram positions
- **Focal Loss** reduces gradient contribution from easy non-suicide samples, focusing training on hard misclassified cases

In [ ]:
# ── Sub-module 1: Attention Gate ──────────────────────────────────
class AttentionGate(nn.Module):
    """
    Soft attention over CNN feature map positions.
    Learns which n-gram locations are most indicative of suicidal ideation.

    Forward:
        x: (B, num_filters, seq_len) after Conv1d
        → permute → (B, seq_len, num_filters)
        → linear score per position → softmax weights
        → weighted sum → (B, num_filters)  context vector
    """
    def __init__(self, feature_dim: int):
        super().__init__()
        self.attn = nn.Linear(feature_dim, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x       = x.permute(0, 2, 1)              # (B, seq, filters)
        scores  = self.attn(x)                     # (B, seq, 1)
        weights = torch.softmax(scores, dim=1)     # (B, seq, 1)
        context = (weights * x).sum(dim=1)         # (B, filters)
        return context


# ── Sub-module 2: Multi-Scale CNN ─────────────────────────────────
class MultiScaleCNN(nn.Module):
    """
    Three parallel 1D convolutions with kernel sizes 2, 3, 4.
    Each kernel targets different n-gram window lengths:
      k=2 → bigrams  (e.g. 'kill myself', 'end it')
      k=3 → trigrams (e.g. 'want to die')
      k=4 → 4-grams  (e.g. 'going to end it')
    Each branch has its own Attention Gate and BatchNorm.
    Output: concatenation of all branches → (B, filters * num_kernels)
    """
    def __init__(self, in_channels: int, out_channels: int,
                 kernel_sizes=CNN_KERNEL_SIZES):
        super().__init__()
        self.convs = nn.ModuleList([
            nn.Conv1d(in_channels, out_channels, kernel_size=k, padding=k//2)
            for k in kernel_sizes
        ])
        self.gates = nn.ModuleList([AttentionGate(out_channels) for _ in kernel_sizes])
        self.bns   = nn.ModuleList([nn.BatchNorm1d(out_channels) for _ in kernel_sizes])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, seq_len, hidden) → (B, hidden, seq_len) for Conv1d
        x = x.permute(0, 2, 1)
        out = []
        for conv, gate, bn in zip(self.convs, self.gates, self.bns):
            h = F.gelu(bn(conv(x)))   # (B, filters, seq')
            out.append(gate(h))       # (B, filters) attended context
        return torch.cat(out, dim=-1) # (B, filters * num_kernels)


# ── Main Proposed Model ────────────────────────────────────────────
class BertCNNClassifier(nn.Module):
    """
    PROPOSED NOVEL ARCHITECTURE for suicidal ideation detection.
    Combines BERT's global contextual understanding with local
    phrase-level features extracted by a multi-scale CNN + attention gate.
    Trained with Focal Loss for better handling of class imbalance.
    """
    def __init__(self, model_name: str = "bert-base-uncased",
                 num_classes: int = 2,
                 cnn_filters: int = CNN_FILTERS,
                 kernel_sizes=CNN_KERNEL_SIZES,
                 dropout: float = DROPOUT_RATE):
        super().__init__()
        self.encoder    = AutoModel.from_pretrained(model_name)
        hidden_size     = self.encoder.config.hidden_size   # 768
        cnn_out_dim     = cnn_filters * len(kernel_sizes)  # 256*3 = 768
        fusion_dim      = hidden_size + cnn_out_dim         # 1536

        self.cnn        = MultiScaleCNN(hidden_size, cnn_filters, kernel_sizes)
        self.layer_norm = nn.LayerNorm(fusion_dim)
        self.dropout    = nn.Dropout(dropout)

        # Deep MLP classifier head
        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim, 512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(512, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes),
        )

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        out = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            output_hidden_states=False,
        )

        # ── Global branch: [CLS] token ──────────────────────
        if hasattr(out, "pooler_output") and out.pooler_output is not None:
            cls_vec = out.pooler_output
        else:
            cls_vec = out.last_hidden_state[:, 0, :]

        # ── Local branch: multi-scale CNN ───────────────────
        cnn_vec = self.cnn(out.last_hidden_state)

        # ── Residual fusion ─────────────────────────────────
        fused = torch.cat([cls_vec, cnn_vec], dim=-1)  # (B, 1536)
        fused = self.layer_norm(fused)
        fused = self.dropout(fused)

        return self.classifier(fused)


print("All model classes defined.")
m = BertCNNClassifier()
t, tr = count_params(m)
print(f"\nBERT-CNN params : {t/1e6:.1f}M total | {tr/1e6:.1f}M trainable")
del m

## 3.3 Model Factory

In [ ]:
def get_model(model_key: str) -> nn.Module:
    """Return the appropriate model for a given key and move to device."""
    ckpt = MODELS[model_key]
    if model_key == "BERT-CNN":
        model = BertCNNClassifier(model_name=ckpt)
    else:
        model = BaselineClassifier(model_name=ckpt)
    return model.to(DEVICE)

# Summary table
print(f"{'Model':<12} {'Checkpoint':<30} {'Architecture'}")
print("-"*65)
for k, v in MODELS.items():
    arch = "BERT-CNN Hybrid (proposed)" if k=="BERT-CNN" else "Baseline [CLS] + Linear"
    print(f"{k:<12} {v:<30} {arch}")
print("\nModel factory ready. Proceed to Notebook 4 for training.")